# CS1-EXP0 — Within-Project StratifiedKFold Comparison

## Purpose

This notebook repeats the **same frozen EXP-0 lexical Logistic Regression**
configuration using a different, explicitly secondary validation protocol:

```text
StratifiedKFold, shuffle=True, random_state=42
```

### Interpretation

| Protocol | Question answered | Project overlap |
|---|---|---:|
| Primary: `StratifiedGroupKFold(project)` | Can the model generalize to unseen projects? | No |
| This notebook: `StratifiedKFold` | How well does it perform when training contains functions from the same project ecosystems? | Yes, deliberate |

This notebook must **not** overwrite or replace the grouped EXP-0 result.

Only the fold assignment changes. Dataset, normalization, TF-IDF settings,
SGD Logistic Regression settings, seed, threshold, and metrics remain the same.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Define paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

EXP0_OUTPUT_ROOT = OUTPUT_ROOT / "cs1_exp0_lr"
EXP0_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

WITHIN_PROJECT_MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_within_project_stratified_5fold_manifest.parquet"
)

GROUPED_MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_project_grouped_5fold_manifest.parquet"
)

print("Normalized dataset:", NORMALIZED_DATA_PATH)
print("New within-project manifest:", WITHIN_PROJECT_MANIFEST_PATH)
print("Original grouped manifest:", GROUPED_MANIFEST_PATH)
print("EXP-0 output root:", EXP0_OUTPUT_ROOT)

Normalized dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
New within-project manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_within_project_stratified_5fold_manifest.parquet
Original grouped manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_grouped_5fold_manifest.parquet
EXP-0 output root: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_exp0_lr


## 3. Clone or refresh the `prashant` branch

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(PROJECT_DIR)

print("Repository:", REPO_DIR)
print("Current branch:", BRANCH)
print("Working directory:", Path.cwd())

Cloning into '/content/DiverseVul--IS-Project'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 56 (delta 7), reused 45 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 602.28 KiB | 6.84 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Repository: /content/DiverseVul--IS-Project
Current branch: prashant
Working directory: /content/DiverseVul--IS-Project/vuln-detection


## 4. Install dependencies

In [ ]:
!pip -q install \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    matplotlib \
    pyarrow \
    joblib

## 5. Import the within-project manifest and EXP-0 runner

Ensure the repository contains these new modules:

```text
src/case_study_1/within_project_manifest.py
src/case_study_1/exp0_lr_within_project.py
```


In [ ]:
import pandas as pd
import numpy as np

import case_study_1.within_project_manifest as within_project_manifest
import case_study_1.exp0_lr_within_project as exp0_wp
import case_study_1.evaluation as evaluation

print("Manifest version:", within_project_manifest.MANIFEST_VERSION)
print("EXP-0 version:", exp0_wp.EXP0_VERSION)

Manifest version: cs1-within-project-stratified-5fold-v1
EXP-0 version: cs1-exp0-sgd-logistic-v1-within-project


## 6. Load the frozen normalized dataset

In [ ]:
if not NORMALIZED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Normalized dataset not found: {NORMALIZED_DATA_PATH}"
    )

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)

assert len(normalized_df) == 261_667
assert normalized_df["source_row_id"].nunique() == len(normalized_df)
assert set(normalized_df["label"].unique()) == {0, 1}
assert normalized_df["normalized_code"].notna().all()

print("Normalized dataset shape:", normalized_df.shape)
print("Projects:", normalized_df["project"].nunique())
print("Vulnerable rate:", f"{normalized_df['label'].mean():.4%}")

Normalized dataset shape: (261667, 5)
Projects: 797
Vulnerable rate: 5.3266%


## 7. Create or load the frozen within-project StratifiedKFold manifest

This manifest has the same schema as the grouped manifest but deliberately
allows project overlap between training and test partitions.


In [ ]:
within_config = within_project_manifest.WithinProjectSplitConfig(
    n_splits=5,
    random_state=42,
    shuffle=True,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
)

if WITHIN_PROJECT_MANIFEST_PATH.exists():
    within_manifest_df = (
        within_project_manifest.load_within_project_manifest(
            WITHIN_PROJECT_MANIFEST_PATH,
            config=within_config,
        )
    )
    print("✅ Loaded existing within-project StratifiedKFold manifest.")
else:
    print("Creating frozen within-project StratifiedKFold manifest...")

    within_manifest_df = (
        within_project_manifest.create_within_project_stratified_manifest(
            normalized_df[
                [
                    "source_row_id",
                    "label",
                    "project",
                ]
            ],
            config=within_config,
        )
    )

    manifest_artifacts = (
        within_project_manifest.save_within_project_manifest_artifacts(
            manifest=within_manifest_df,
            output_dir=MANIFEST_DIR,
            config=within_config,
            normalized_dataset_path=NORMALIZED_DATA_PATH,
        )
    )

    print("✅ Manifest created and saved:")
    for artifact_name, artifact_path in manifest_artifacts.__dict__.items():
        print(f" - {artifact_name}: {artifact_path}")

print("Manifest shape:", within_manifest_df.shape)
display(within_manifest_df.head())

Creating frozen within-project StratifiedKFold manifest...
✅ Manifest created and saved:
 - csv_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_within_project_stratified_5fold_manifest.csv
 - parquet_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_within_project_stratified_5fold_manifest.parquet
 - summary_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_within_project_stratified_5fold_fold_summary.csv
 - metadata_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_within_project_stratified_5fold_metadata.json
Manifest shape: (261667, 4)


,source_row_id,label,project,fold
0,0,0,linux,1
1,1,0,net,0
2,2,0,net,3
3,3,0,net,1
4,4,0,net,1


## 8. Audit the intended project overlap

This is the core difference from grouped CV. High overlap is expected here and
is the reason this protocol measures **within-project** rather than unseen-project
generalization.


In [ ]:
within_summary_df = (
    within_project_manifest.summarize_within_project_manifest(
        within_manifest_df,
        config=within_config,
    )
)

display(
    within_summary_df.style.format(
        {
            "test_positive_rate": "{:.4%}",
            "positive_rate_delta_from_global": "{:+.4%}",
            "overlap_rate_of_test_projects": "{:.2%}",
            "test_row_overlap_rate": "{:.2%}",
            "test_row_share": "{:.2%}",
            "test_project_share": "{:.2%}",
        }
    )
)

assert (
    within_summary_df["test_row_overlap_rate"] > 0.95
).all(), (
    "Unexpectedly low project overlap. "
    "Check that the correct StratifiedKFold manifest was created."
)

print("✅ Project overlap is present as intended.")
print(
    "Mean test rows from projects also present in training:",
    f"{within_summary_df['test_row_overlap_rate'].mean():.2%}",
)

,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,overlap_rate_of_test_projects,test_rows_from_overlapping_projects,test_row_overlap_rate,test_row_share,test_project_share
0,0,52334,2788,49546,5.3273%,+0.0007%,760,209333,797,760,100.00%,52334,100.00%,20.00%,95.36%
1,1,52334,2788,49546,5.3273%,+0.0007%,765,209333,795,763,99.74%,52331,99.99%,20.00%,95.98%
2,2,52333,2787,49546,5.3255%,-0.0011%,766,209334,795,764,99.74%,52331,100.00%,20.00%,96.11%
3,3,52333,2787,49546,5.3255%,-0.0011%,769,209334,795,767,99.74%,52327,99.99%,20.00%,96.49%
4,4,52333,2788,49545,5.3274%,+0.0008%,770,209334,796,769,99.87%,52331,100.00%,20.00%,96.61%


✅ Project overlap is present as intended.
Mean test rows from projects also present in training: 100.00%


## 9. Compare fold characteristics with the strict grouped manifest

This is a protocol audit, not a model result.


In [ ]:
if GROUPED_MANIFEST_PATH.exists():
    grouped_manifest_df = pd.read_parquet(GROUPED_MANIFEST_PATH)

    grouped_rows = []

    for fold_id in range(5):
        test_projects = set(
            grouped_manifest_df.loc[
                grouped_manifest_df["fold"] == fold_id,
                "project",
            ]
        )
        train_projects = set(
            grouped_manifest_df.loc[
                grouped_manifest_df["fold"] != fold_id,
                "project",
            ]
        )

        grouped_rows.append(
            {
                "protocol": "cross_project_grouped",
                "fold": fold_id,
                "test_rows": int(
                    (grouped_manifest_df["fold"] == fold_id).sum()
                ),
                "test_unique_projects": len(test_projects),
                "train_test_project_overlap": len(
                    test_projects.intersection(train_projects)
                ),
            }
        )

    grouped_audit_df = pd.DataFrame(grouped_rows)

    within_audit_df = within_summary_df[
        [
            "fold",
            "test_rows",
            "test_unique_projects",
            "train_test_project_overlap",
        ]
    ].copy()

    within_audit_df.insert(
        0,
        "protocol",
        "within_project_stratified",
    )

    protocol_audit_df = pd.concat(
        [grouped_audit_df, within_audit_df],
        ignore_index=True,
    )

    display(protocol_audit_df)
else:
    print(
        "Grouped manifest not found in Drive. "
        "The within-project manifest itself is still valid."
    )

,protocol,fold,test_rows,test_unique_projects,train_test_project_overlap
0,cross_project_grouped,0,69256,54,0
1,cross_project_grouped,1,51771,188,0
2,cross_project_grouped,2,46043,192,0
3,cross_project_grouped,3,47064,181,0
4,cross_project_grouped,4,47533,182,0
5,within_project_stratified,0,52334,760,760
6,within_project_stratified,1,52334,765,763
7,within_project_stratified,2,52333,766,764
8,within_project_stratified,3,52333,769,767
9,within_project_stratified,4,52333,770,769


## 10. Declare the identical EXP-0 model configuration

Do not tune these values after seeing the StratifiedKFold results. They are
identical to the frozen grouped EXP-0 configuration.


In [ ]:
exp0_within_config = exp0_wp.Exp0Config(
    experiment_name="cs1_exp0_lr_within_project_stratified",

    n_splits=5,
    random_state=42,
    decision_threshold=0.50,

    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,

    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,

    sgd_loss="log_loss",
    sgd_penalty="l2",
    sgd_alpha=1e-5,
    sgd_class_weight="balanced",
    sgd_max_iter=80,
    sgd_tol=1e-3,
    sgd_average=True,

    top_features_per_direction=30,
    verbose=True,
)

print(exp0_within_config)

Exp0Config(experiment_name='cs1_exp0_lr_within_project_stratified', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', n_splits=5, random_state=42, decision_threshold=0.5, word_ngram_range=(1, 3), word_min_df=3, word_max_df=0.995, word_max_features=50000, char_analyzer='char', char_ngram_range=(3, 4), char_min_df=8, char_max_df=0.995, char_max_features=60000, lowercase=False, sublinear_tf=True, tfidf_norm='l2', sgd_loss='log_loss', sgd_penalty='l2', sgd_alpha=1e-05, sgd_class_weight='balanced', sgd_max_iter=80, sgd_tol=0.001, sgd_average=True, top_features_per_direction=30, verbose=True)


## 11. Profile one StratifiedKFold fold

This is a feasibility and protocol check only. Do not interpret the profile
metrics as the official five-fold result.


In [ ]:
exp0_within_profile = exp0_wp.run_exp0_profile_fold(
    normalized_frame=normalized_df,
    manifest=within_manifest_df,
    fold_id=0,
    config=exp0_within_config,
)

print("\nFold 0 computational profile:")
display(
    exp0_within_profile["training_metadata"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_unique_projects",
            "test_unique_projects",
            "train_test_project_overlap",
            "overlap_rate_of_test_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "model_n_iter",
            "convergence_warning_count",
        ]
    ]
)

print("\nFold 0 descriptive metrics — not official final results:")
print(
    evaluation.format_metric_report(
        exp0_within_profile["profile_metrics"]
    )
)

[07:18:35] CS1-EXP0 profiling mode: running Fold 1/5 only.
[07:18:37] Fold 1/5 started | train=209,333, test=52,334, train projects=797, test projects=760.
[07:18:37] Fold 1/5 | fitting word TF-IDF...
[07:21:54] Fold 1/5 | word TF-IDF done in 3.29 min (50,000 features).
[07:21:54] Fold 1/5 | fitting character TF-IDF...
[07:28:08] Fold 1/5 | character TF-IDF done in 6.23 min (60,000 features).
[07:28:08] Fold 1/5 | joining sparse feature matrices...
[07:28:11] Fold 1/5 | sparse matrices ready in 2.5s (total features: 110,000).
[07:28:12] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[07:28:39] Fold 1/5 | SGD Logistic Regression done in 27.8s (epochs=22, convergence warnings=0).
[07:28:39] Fold 1/5 | scoring mixed-project held-out functions...
[07:28:40] Fold 1/5 | scoring done in 0.1s.
[07:28:40] Fold 1/5 completed in 10.05 min.
[07:28:40] Profiling run complete. These metrics are descriptive for one fold only; do not treat them as the official

,fold,train_rows,test_rows,train_unique_projects,test_unique_projects,train_test_project_overlap,overlap_rate_of_test_projects,word_tfidf_seconds,char_tfidf_seconds,model_fit_seconds,total_fold_seconds,model_n_iter,convergence_warning_count
0,0,209333,52334,797,760,760,1.0,197.299726,373.81254,27.80199,603.075161,22,0



Fold 0 descriptive metrics — not official final results:
Pooled Out-of-Fold Evaluation
                   n_samples: 52334
                vulnerable_1: 2788
            non_vulnerable_0: 49546
               positive_rate: 0.053273
                   threshold: 0.500000
    average_precision_pr_auc: 0.248192
                   precision: 0.185276
                      recall: 0.674319
                          f1: 0.290684
                         mcc: 0.288269
                 specificity: 0.833145
         false_positive_rate: 0.166855
               true_negative: 41279
              false_positive: 8267
              false_negative: 908
               true_positive: 1880


## 12. Official five-fold within-project run

Run this cell **only after the profile above completes successfully**.

A fresh output folder prevents accidental mixing with strict grouped results.


In [ ]:
WITHIN_PROJECT_OFFICIAL_OUTPUT_DIR = (
    EXP0_OUTPUT_ROOT / "official_within_project_stratified_v1"
)

WITHIN_PROJECT_OFFICIAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_output_files = list(
    WITHIN_PROJECT_OFFICIAL_OUTPUT_DIR.iterdir()
)

if existing_output_files:
    raise RuntimeError(
        "Official output folder is not empty. "
        "Use a fresh directory or deliberately inspect/remove only incomplete artifacts."
    )

exp0_within_results = exp0_wp.run_exp0(
    normalized_frame=normalized_df,
    manifest=within_manifest_df,
    config=exp0_within_config,
    output_dir=WITHIN_PROJECT_OFFICIAL_OUTPUT_DIR,
    additional_metadata={
        "run_type": "official_full_5fold_oof_evaluation",
        "evaluation_protocol": "within_project_stratified",
        "protocol_interpretation": (
            "Secondary evaluation: functions from the same projects can occur "
            "in train and test folds. This measures within-project performance "
            "and must not be interpreted as unseen-project generalization."
        ),
        "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
        "manifest_path": str(WITHIN_PROJECT_MANIFEST_PATH),
        "model_configuration_control": (
            "Identical to official grouped EXP-0 except for the fold assignment."
        ),
        "primary_comparator": (
            "official_sgd_logistic_v1 using StratifiedGroupKFold by project"
        ),
    },
)

print(
    evaluation.format_metric_report(
        exp0_within_results["evaluation"]["pooled_metrics"]
    )
)

[07:34:22] CS1-EXP0 official run started: 5-fold within-project StratifiedKFold.
[07:34:22] Configuration: word<= 50,000, char<= 60,000, optimizer=SGDClassifier(log_loss), max_iter=80, alpha=1e-05.
[07:34:23] Fold 1/5 started | train=209,333, test=52,334, train projects=797, test projects=760.
[07:34:23] Fold 1/5 | fitting word TF-IDF...
[07:37:49] Fold 1/5 | word TF-IDF done in 3.43 min (50,000 features).
[07:37:49] Fold 1/5 | fitting character TF-IDF...
[07:44:13] Fold 1/5 | character TF-IDF done in 6.40 min (60,000 features).
[07:44:13] Fold 1/5 | joining sparse feature matrices...
[07:44:15] Fold 1/5 | sparse matrices ready in 1.7s (total features: 110,000).
[07:44:15] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[07:44:42] Fold 1/5 | SGD Logistic Regression done in 26.5s (epochs=22, convergence warnings=0).
[07:44:42] Fold 1/5 | scoring mixed-project held-out functions...
[07:44:42] Fold 1/5 | scoring done in 0.1s.
[07:44:42] Fold 1/5 co

## 13. Show the final comparison-ready outputs

Run after Section 12 completes.


In [ ]:
display(
    exp0_within_results["evaluation"]["fold_metrics"][
        [
            "fold",
            "n_samples",
            "test_unique_projects",
            "positive_rate",
            "average_precision_pr_auc",
            "precision",
            "recall",
            "f1",
            "mcc",
            "false_positive_rate",
            "predicted_positive_rate",
        ]
    ]
)

display(
    exp0_within_results["fold_training"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_test_project_overlap",
            "overlap_rate_of_test_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "model_n_iter",
            "convergence_warning_count",
        ]
    ]
)

,fold,n_samples,test_unique_projects,positive_rate,average_precision_pr_auc,precision,recall,f1,mcc,false_positive_rate,predicted_positive_rate
0,0,52334,760,0.053273,0.248192,0.185276,0.674319,0.290684,0.288269,0.166855,0.193889
1,1,52334,765,0.053273,0.256379,0.184321,0.677188,0.289771,0.287860,0.168631,0.195724
2,2,52333,766,0.053255,0.258357,0.182151,0.677431,0.287105,0.285278,0.171094,0.198059
3,3,52333,769,0.053255,0.247146,0.186368,0.672049,0.291813,0.289017,0.165039,0.192039
4,4,52333,770,0.053274,0.243872,0.182565,0.661765,0.286180,0.281638,0.166737,0.193110


,fold,train_rows,test_rows,train_test_project_overlap,overlap_rate_of_test_projects,word_tfidf_seconds,char_tfidf_seconds,model_fit_seconds,total_fold_seconds,model_n_iter,convergence_warning_count
0,0,209333,52334,760,1.000000,205.794915,384.088067,26.488318,619.075402,22,0
1,1,209333,52334,763,0.997386,202.958058,389.656346,21.594004,617.325686,19,0
2,2,209334,52333,764,0.997389,204.125515,389.407695,23.813861,620.146249,19,0
3,3,209334,52333,767,0.997399,204.145331,391.228901,21.743241,620.972661,18,0
4,4,209334,52333,769,0.998701,214.807413,387.106619,27.150031,632.141908,21,0
